# Mach-Zero Backtest Analysis

Run backtests using historical data from QuestDB and analyze results.

In [ ]:
import subprocess
import json
import numpy as np

In [ ]:
# Fetch historical trades from QuestDB REST API
try:
    import urllib.request
    url = 'http://localhost:9000/exec?query=SELECT+*+FROM+trades+ORDER+BY+timestamp+LIMIT+1000'
    with urllib.request.urlopen(url, timeout=5) as resp:
        data = json.loads(resp.read())
        columns = data.get('columns', [])
        dataset = data.get('dataset', [])
        print(f'Fetched {len(dataset)} trades from QuestDB')
        if columns:
            print(f'Columns: {[c["name"] for c in columns]}')
except Exception as e:
    print(f'QuestDB not available: {e}')
    print('Using synthetic data instead')
    dataset = None

In [ ]:
# Generate synthetic price series for backtesting
np.random.seed(42)
n_ticks = 10000
base_price = 50000.0

# Geometric Brownian Motion
dt = 1.0 / n_ticks
mu = 0.0
sigma = 0.02
returns = np.random.normal(mu * dt, sigma * np.sqrt(dt), n_ticks)
prices = base_price * np.exp(np.cumsum(returns))

print(f'Price range: {prices.min():.2f} - {prices.max():.2f}')
print(f'Final price: {prices[-1]:.2f}')
print(f'Total return: {(prices[-1]/prices[0] - 1)*100:.2f}%')

In [ ]:
# Simple spread strategy simulation (pure Python)
spread_offset = 1.0  # Place orders 1.0 away from mid
position = 0.0
pnl = 0.0
trades = []
equity_curve = [0.0]

for i in range(1, len(prices)):
    mid = prices[i]
    buy_price = mid - spread_offset
    sell_price = mid + spread_offset
    
    # Simple mean-reversion: buy on dip, sell on rise
    ret = prices[i] / prices[i-1] - 1
    
    if ret < -0.001 and position <= 0:  # Dip -> buy
        position += 1.0
        pnl -= buy_price
        trades.append(('BUY', buy_price, i))
    elif ret > 0.001 and position >= 1:  # Rise -> sell
        position -= 1.0
        pnl += sell_price
        trades.append(('SELL', sell_price, i))
    
    # Mark-to-market
    mtm = pnl + position * mid
    equity_curve.append(mtm)

# Close final position
if position != 0:
    pnl += position * prices[-1]
    equity_curve[-1] = pnl

print(f'Total trades: {len(trades)}')
print(f'Final P&L: ${pnl:.2f}')
print(f'Max equity: ${max(equity_curve):.2f}')
print(f'Min equity: ${min(equity_curve):.2f}')

In [ ]:
# Compute performance stats
equity = np.array(equity_curve)
returns_eq = np.diff(equity)
returns_eq = returns_eq[returns_eq != 0]  # Filter zero returns

if len(returns_eq) > 1:
    sharpe = np.mean(returns_eq) / np.std(returns_eq) * np.sqrt(252) if np.std(returns_eq) > 0 else 0
    
    # Max drawdown
    peak = np.maximum.accumulate(equity)
    drawdown = peak - equity
    max_dd = drawdown.max()
    
    wins = sum(1 for t in trades if t[0] == 'SELL')
    
    print(f'Sharpe Ratio: {sharpe:.2f}')
    print(f'Max Drawdown: ${max_dd:.2f}')
    print(f'Win Rate: {wins/max(len(trades),1)*100:.1f}%')
else:
    print('Not enough returns to compute stats')

In [ ]:
# Plot equity curve
try:
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    
    # Price chart with trade markers
    axes[0].plot(prices, 'k-', linewidth=0.5, alpha=0.7, label='Price')
    buys = [(t[2], t[1]) for t in trades if t[0] == 'BUY']
    sells = [(t[2], t[1]) for t in trades if t[0] == 'SELL']
    if buys:
        axes[0].scatter([b[0] for b in buys], [b[1] for b in buys],
                       c='green', marker='^', s=15, alpha=0.6, label='Buy')
    if sells:
        axes[0].scatter([s[0] for s in sells], [s[1] for s in sells],
                       c='red', marker='v', s=15, alpha=0.6, label='Sell')
    axes[0].set_ylabel('Price ($)')
    axes[0].set_title('Backtest: Price & Trades')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Equity curve
    axes[1].plot(equity_curve, 'b-', linewidth=0.8)
    axes[1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    axes[1].fill_between(range(len(equity_curve)), equity_curve, 0,
                        where=[e >= 0 for e in equity_curve], alpha=0.1, color='green')
    axes[1].fill_between(range(len(equity_curve)), equity_curve, 0,
                        where=[e < 0 for e in equity_curve], alpha=0.1, color='red')
    axes[1].set_xlabel('Tick')
    axes[1].set_ylabel('P&L ($)')
    axes[1].set_title('Equity Curve')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
except ImportError:
    print('matplotlib not installed, skipping plots')